<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/MC10_Advanced_RAG_HR_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MC10: Advanced RAG on our HR policy PDFs
### LlamaParse + four chunking strategies, ready for Google Colab

**Before you run anything**
1. In Colab, open the key icon in the left sidebar (Secrets) and add `OPENAI_API_KEY` and `LLAMA_CLOUD_API_KEY`. Switch on "Notebook access" for both. No key is typed into any cell.
2. Put your HR PDF(s) in a folder in Colab (Files panel, left sidebar) and set `PDF_DIR` in the configuration cell.
3. Run the cells from top to bottom.

```
PDF folder --> LlamaParse (markdown) --> 4 chunking strategies --> one index each --> compare --> Gradio app
```

## 1. Setup

In [1]:
!pip install -q llama-index-core llama-index-llms-openai llama-index-embeddings-openai "llama-cloud>=2.8" pandas gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 483.7/483.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.7/168.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 8.8 MB/s eta 0:00:00


In [2]:
# API keys come from Colab Secrets. Nothing is hardcoded and nothing is printed.
import os
from google.colab import userdata
#make sure name is same as the secret u added
for name in ("openai", "llama"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: found")
    except Exception:
        raise RuntimeError(f"Add {name} in Colab Secrets (key icon, left sidebar) and switch on Notebook access.")

openai: found
llama: found


In [3]:
# ---- Configuration ----
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

PDF_DIR     = "/content/pdf"          # folder that holds your HR PDF(s)
PARSED_DIR  = "/content/parsed_md"    # parsed markdown is saved here and reused (saves LlamaParse credits)
PARSE_TIER  = "agentic"               # "cost_effective" | "agentic" | "agentic_plus"
LLM_MODEL   = "gpt-4o-mini"           # change to the latest OpenAI model you use
EMBED_MODEL = "text-embedding-3-small"
TOP_K       = 3                       # chunks retrieved per question

Settings.llm = OpenAI(model=LLM_MODEL, temperature=0)
Settings.embed_model = OpenAIEmbedding(model=EMBED_MODEL)
print("Configured:", LLM_MODEL, "|", EMBED_MODEL, "| top_k =", TOP_K)

Configured: gpt-4o-mini | text-embedding-3-small | top_k = 3


## 2. Parse the PDF folder with LlamaParse

In [6]:
import os
from google.colab import userdata
os.environ["LLAMA_CLOUD_API_KEY"] = userdata.get('llama')


In [7]:
import glob, time
from pathlib import Path
from llama_cloud import LlamaCloud

pdf_files = sorted(glob.glob(f"{PDF_DIR}/*.pdf"))
assert pdf_files, f"No PDF files found in {PDF_DIR}. Upload your PDF(s) there or change PDF_DIR."
Path(PARSED_DIR).mkdir(parents=True, exist_ok=True)
client = LlamaCloud()          # reads LLAMA_CLOUD_API_KEY from the environment

def parse_pdf(path):
    uploaded = client.files.create(file=path, purpose="parse")
    result = client.parsing.parse(
        file_id=uploaded.id, tier=PARSE_TIER, version="latest",
        output_options={"markdown": {"tables": {"output_tables_as_markdown": True}}},
        expand=["markdown"],
    )
    return "\n\n".join((getattr(p, "markdown", "") or "") for p in result.markdown.pages)

DOCS = {}                       # document name -> markdown
for path in pdf_files:
    stem = Path(path).stem
    cache = Path(PARSED_DIR) / f"{stem}.md"
    if cache.exists():
        md_text, how = cache.read_text(), "reused saved markdown"
    else:
        t0 = time.time()
        md_text = parse_pdf(path)
        cache.write_text(md_text)
        how = f"parsed with LlamaParse ({PARSE_TIER}) in {time.time() - t0:.0f}s"
    DOCS[stem] = md_text
    print(f"{stem}: {len(md_text):,} characters, {how}")

demo_name = next(iter(DOCS))                      # first document, used to preview each strategy below
demo_md = DOCS[demo_name]
print("\n--- first 700 characters of the first document ---")
print(demo_md[:700])

leave_absence_policy_v3.1: 12,554 characters, parsed with LlamaParse (agentic) in 25s

--- first 700 characters of the first document ---
# MERIDIAN GROUP

DUBAI · ABU DHABI · RIYADH · CAIRO · BENGALURU

LEAVE

## Leave and Absence Policy

Annual, sick, parental and special leave entitlements across all Meridian Group entities.

| Document code    | POL-HR-002                   |
| ---------------- | ---------------------------- |
| Version          | 3.1                          |
| Effective date   | 2026-01-01                   |
| Next review date | 2027-01-01                   |
| Document owner   | Head of Human Resources      |
| Applies to       | All employees, all locations |
| Classification   | Internal                     |

## Document Control

This policy is maintained by the document owner named on the cover pa


## 3. Four chunking strategies
Every strategy gets the same LlamaParse markdown, so the comparison is fair.

| # | Strategy | Idea |
|---|---|---|
| 1 | Fixed-size | Cut every N characters, with a small overlap |
| 2 | Recursive | Split on paragraphs, then lines, then sentences, then words |
| 3 | Semantic | Embed sentences and cut where the meaning shifts |
| 4 | Table-aware | Follow markdown headings, keep each table whole, label every chunk with its section |

In [8]:
import re
import numpy as np
import pandas as pd

def norm(s):
    """Lowercase and collapse everything that is not a letter or digit. Used for matching."""
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

def preview(chunks, n=2, width=420):
    print(f"{len(chunks)} chunks | average {sum(len(c) for c in chunks) // len(chunks)} characters\n")
    for i, c in enumerate(chunks[:n]):
        print(f"--- chunk {i} ({len(c)} chars) ---")
        print(c[:width] + (" ..." if len(c) > width else ""))
        print()

### Strategy 1: Fixed-size

In [9]:
def chunk_fixed(text, size=450, overlap=50):
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step) if text[i:i + size].strip()]

fixed_chunks = chunk_fixed(demo_md)
preview(fixed_chunks)

32 chunks | average 440 characters

--- chunk 0 (450 chars) ---
# MERIDIAN GROUP

DUBAI · ABU DHABI · RIYADH · CAIRO · BENGALURU

LEAVE

## Leave and Absence Policy

Annual, sick, parental and special leave entitlements across all Meridian Group entities.

| Document code    | POL-HR-002                   |
| ---------------- | ---------------------------- |
| Version          | 3.1                          |
| Effective date   | 2026-01-01                   |
| Next review date  ...

--- chunk 1 (450 chars) ---

| Next review date | 2027-01-01                   |
| Document owner   | Head of Human Resources      |
| Applies to       | All employees, all locations |
| Classification   | Internal                     |

## Document Control

This policy is maintained by the document owner named on the cover page. It is reviewed at least annually, and earlier where legislation, regulation or business circumstances require. Emplo ...



### Strategy 2: Recursive

In [ ]:
def chunk_recursive(text, size=450, overlap=50, separators=("\n\n", "\n", ". ", " ", "")):
    def merge(pieces, sep):
        chunks, cur = [], []
        for p in pieces:
            if cur and len(sep.join(cur + [p])) > size:
                chunks.append(sep.join(cur))
                while cur and (len(sep.join(cur)) > overlap or len(sep.join(cur + [p])) > size):
                    cur.pop(0)
            cur.append(p)
        if cur:
            chunks.append(sep.join(cur))
        return chunks

    def split(t, seps):
        sep = next((s for s in seps if s == "" or s in t), "")
        rest = seps[seps.index(sep) + 1:]
        parts = t.split(sep) if sep else list(t)
        out, good = [], []
        for part in parts:
            if len(part) <= size:
                good.append(part)
            else:
                if good:
                    out += merge(good, sep); good = []
                out += split(part, rest) if rest else [part[i:i + size] for i in range(0, len(part), size)]
        if good:
            out += merge(good, sep)
        return out

    return [c for c in split(text, list(separators)) if c.strip()]

recursive_chunks = chunk_recursive(demo_md)
preview(recursive_chunks)

### Strategy 3: Semantic

In [ ]:
def split_units(text):
    """Turn text into sentence-like units. Headings, table rows and short lines stay as single units."""
    units = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith(("|", "#", "*", "!")) or len(line) < 80:
            units.append(line)
        else:
            units += [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", line) if s.strip()]
    return units

def chunk_semantic(text, percentile=85, min_chars=150, max_chars=900, buffer=1):
    units = split_units(text)
    if len(units) < 3:
        return ["\n".join(units)] if units else []
    windows = [" ".join(units[max(0, i - buffer): i + buffer + 1]) for i in range(len(units))]
    vecs = np.array(Settings.embed_model.get_text_embedding_batch(windows))
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    distance = 1 - (vecs[:-1] * vecs[1:]).sum(axis=1)          # meaning shift between neighbouring units
    cut = np.percentile(distance, percentile)                  # only the biggest shifts become boundaries
    chunks, cur = [], [units[0]]
    for unit, d in zip(units[1:], distance):
        size_now = len("\n".join(cur))
        if (d > cut and size_now >= min_chars) or size_now + len(unit) > max_chars:
            chunks.append("\n".join(cur)); cur = [unit]
        else:
            cur.append(unit)
    chunks.append("\n".join(cur))
    return chunks

semantic_chunks = chunk_semantic(demo_md)
preview(semantic_chunks)

### Strategy 4: Table-aware

In [ ]:
def chunk_table_aware(md, max_chars=700, doc=None):
    heading_re = re.compile(r"^(#{1,6})\s+(.*)$")
    caption_re = re.compile(r"^(\*[^*]+\*|Table\s+\d+.*)$")
    blocks, stack, para = [], [], []
    single_title = len(re.findall(r"^# ", md, flags=re.M)) == 1       # one H1 = the document title, keep it out of the path

    def path():
        return " > ".join(t for _, t in stack) or "Document"

    def flush():
        if para:
            blocks.append(("text", "\n".join(para).strip(), path()))
            para.clear()

    lines, i = md.split("\n"), 0
    while i < len(lines):
        s = lines[i].strip()
        m = heading_re.match(s)
        if m:                                                   # a heading updates the section path
            flush()
            level = len(m.group(1))
            if not (level == 1 and single_title):
                while stack and stack[-1][0] >= level:
                    stack.pop()
                stack.append((level, m.group(2).strip()))
        elif s.startswith("|") or s.lower().startswith("<table"):   # a table: collect every line of it
            flush()
            caption = ""
            if blocks and blocks[-1][0] == "text" and "\n" not in blocks[-1][1] and caption_re.match(blocks[-1][1]):
                caption = blocks.pop()[1]                       # the caption line just above the table
            tbl = []
            if s.startswith("|"):
                while i < len(lines) and lines[i].strip().startswith("|"):
                    tbl.append(lines[i].rstrip()); i += 1
            else:                                               # HTML table, in case the parser returns one
                while i < len(lines):
                    tbl.append(lines[i].rstrip())
                    i += 1
                    if "</table>" in tbl[-1].lower():
                        break
            blocks.append(("table", "\n".join(([caption] if caption else []) + tbl), path()))
            continue
        elif s == "":
            flush()
        else:
            para.append(lines[i])
        i += 1
    flush()

    def label(chunks, p):
        where = (doc if p == "Document" else f"{doc} > {p}") if doc else p            # document name + heading path
        return [f"[Section: {where}]\n{c}" for c in chunks if c.strip()]

    def table_chunks(text, p):
        if len(text) <= max_chars:
            return label([text], p)
        rows = text.split("\n")
        head_n = next((k for k, l in enumerate(rows) if l.strip().startswith("|")), 0) + 2   # caption + header + separator
        head, body, out, cur = rows[:head_n], rows[head_n:], [], []
        for r in body:
            if cur and len("\n".join(head + cur + [r])) > max_chars:
                out.append("\n".join(head + cur)); cur = []    # repeat the header in every piece
            cur.append(r)
        if cur:
            out.append("\n".join(head + cur))
        return label(out, p)

    chunks, buf, buf_path = [], [], None

    def emit_text():
        nonlocal buf
        if buf:
            chunks.extend(label(chunk_recursive("\n\n".join(buf), size=max_chars, overlap=0), buf_path))
            buf = []

    for kind, text, p in blocks:
        if not text.strip():
            continue
        if kind == "table":
            emit_text()
            chunks.extend(table_chunks(text, p))
        else:
            if buf and (p != buf_path or len("\n\n".join(buf + [text])) > max_chars):
                emit_text()
            if not buf:
                buf_path = p
            buf.append(text)
    emit_text()
    return chunks

table_aware_chunks = chunk_table_aware(demo_md, doc=demo_name)
preview(table_aware_chunks, n=3, width=520)

## 4. Compare the chunks
Each strategy is applied to every document. **Table rows kept whole** and **tables kept whole** show how well each strategy respects the tables in your PDFs.

In [ ]:
def extract_tables(md):
    """Return every markdown table as a list of rows (cells joined by spaces)."""
    tables, cur = [], []
    for line in md.split("\n") + [""]:
        if line.strip().startswith("|"):
            cells = [c.strip() for c in line.strip().strip("|").split("|")]
            if all(re.fullmatch(r":?-{3,}:?", c) for c in cells):
                continue
            cur.append(" ".join(cells))
        elif cur:
            tables.append(cur); cur = []
    return tables

MD_TABLES = extract_tables("\n\n".join(DOCS.values()))

def build_chunks(chunker, named=False):
    out = []
    for name, doc_md in DOCS.items():
        out += chunker(doc_md, doc=name) if named else chunker(doc_md)
    return out

CHUNKS = {
    "1. Fixed-size":  build_chunks(chunk_fixed),
    "2. Recursive":   build_chunks(chunk_recursive),
    "3. Semantic":    build_chunks(chunk_semantic),
    "4. Table-aware": build_chunks(chunk_table_aware, named=True),   # adds the document name to every chunk label
}

def chunk_report(chunks):
    lens = [len(c) for c in chunks]
    all_rows = [r for t in MD_TABLES for r in t]
    rows_ok = sum(any(norm(r) in norm(c) for c in chunks) for r in all_rows)
    tables_ok = sum(any(all(norm(r) in norm(c) for r in t) for c in chunks) for t in MD_TABLES)
    return {"chunks": len(chunks), "avg chars": round(sum(lens) / len(lens)), "min": min(lens), "max": max(lens),
            "table rows kept whole": f"{rows_ok}/{len(all_rows)}", "tables kept whole": f"{tables_ok}/{len(MD_TABLES)}"}

pd.DataFrame({name: chunk_report(ch) for name, ch in CHUNKS.items()}).T

### Print the chunks
Pick any phrase from a table row in your document and see which chunks contain it, for every strategy. Look for rows that lost their header, or were cut in half.

In [ ]:
def inspect_chunks(needle, width=600):
    for name, chunks in CHUNKS.items():
        hits = [c for c in chunks if norm(needle) in norm(c)]
        print("=" * 90)
        print(f"{name}: {len(hits)} chunk(s) contain '{needle}'")
        for c in hits:
            print("-" * 90)
            print(c[:width])
        if not hits:
            print("   (none: the words were split across two chunks)")

NEEDLE = "half pay"      # change this to any phrase from a table row in your PDF
inspect_chunks(NEEDLE)

## 5. One index per strategy, and the same question on each

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import TextNode

INDEXES = {}
for name, chunks in CHUNKS.items():
    nodes = [TextNode(text=c) for c in chunks]
    INDEXES[name] = VectorStoreIndex(nodes)                     # embeds every chunk with Settings.embed_model
    print(f"indexed {len(nodes):>3} chunks  |  {name}")

In [ ]:
from llama_index.core import PromptTemplate

QA_PROMPT = PromptTemplate(
    "You answer questions about our HR policy documents using ONLY the context below.\n"
    "If the context does not contain the answer, say you cannot find it in the document.\n"
    "Mention the section name when the context provides one.\n\n"
    "Context:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

def ask(strategy, question):
    engine = INDEXES[strategy].as_query_engine(similarity_top_k=TOP_K, text_qa_template=QA_PROMPT)
    response = engine.query(question)
    return str(response).strip(), [n.node.get_content(metadata_mode="none") for n in response.source_nodes]

def compare_answers(question):
    print("QUESTION:", question, "\n")
    for name in INDEXES:
        answer, chunks = ask(name, question)
        print(f"[{name}]")
        print("  answer:", answer.replace("\n", " "))
        print(f"  retrieved {len(chunks)} chunks, {sum(len(c) for c in chunks)} characters\n")

QUESTIONS = [
    "How many days of annual leave do employees get?",
    "What is the sick leave policy?",
    "What is the parental leave policy?",
]      # replace with questions about your own HR documents
for q in QUESTIONS:
    compare_answers(q)

## 6. Test it live with Gradio

In [ ]:
import gradio as gr

def gradio_ask(strategy, question):
    if not question.strip():
        return "Please type a question.", ""
    answer, chunks = ask(strategy, question)
    shown = "\n\n".join(f"**Retrieved chunk {i + 1}** ({len(c)} characters)\n\n```\n{c}\n```" for i, c in enumerate(chunks))
    return answer, shown

with gr.Blocks(title="Advanced RAG: chunking lab") as demo:
    gr.Markdown("# Advanced RAG: chunking lab\nSame document, same model. Only the chunking strategy changes.")
    strategy = gr.Dropdown(choices=list(INDEXES), value="4. Table-aware", label="Chunking strategy")
    question = gr.Textbox(label="Your question", placeholder="How many days of sick leave are paid at half pay?")
    ask_btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Textbox(label="Answer", lines=4)
    chunks_box = gr.Markdown(label="Retrieved chunks")
    gr.Examples(
        examples=[[q] for q in QUESTIONS],
        inputs=[question],
    )
    ask_btn.click(gradio_ask, inputs=[strategy, question], outputs=[answer_box, chunks_box])
    question.submit(gradio_ask, inputs=[strategy, question], outputs=[answer_box, chunks_box])

demo.launch(share=True)     # share=True gives a public link the audience can open on their phones